# 02 — Data Quality Audit & Cleaning

**Objective:** Execute a rigorous data quality audit and produce clean, analysis-ready tables.

This notebook:
1. Runs the `build_dataset_v3.py` ETL pipeline to generate cleaned parquet files
2. Validates output quality with quantitative checks
3. Visualizes data completeness and missingness patterns

**Outputs:**
- `processed_v3/dispatch_records_v3.parquet`
- `processed_v3/mission_records_v3.parquet`
- `processed_v3/aed_records_v3.parquet`

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path('.')
V3_DIR = PROJECT_ROOT / 'mda_project' / 'data' / 'processed_v3'
OUT_DIR = PROJECT_ROOT / 'mda_project' / 'data' / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 2.1 Load Cleaned Datasets

The ETL pipeline (`build_dataset_v3.py`) handles:
- Mixed timestamp format parsing (custom + ISO)
- Geographic bounding box filtering (Belgium extent)
- Outlier removal for response times (> 4 hours excluded)
- Nearest-AED distance calculation via Haversine

In [ ]:
# Load pre-processed v3 datasets
dispatch = pd.read_parquet(V3_DIR / 'dispatch_records_v3.parquet')
mission = pd.read_parquet(V3_DIR / 'mission_records_v3.parquet')
aed = pd.read_parquet(V3_DIR / 'aed_records_v3.parquet')

print(f'Dispatch records: {dispatch.shape[0]:,} rows x {dispatch.shape[1]} columns')
print(f'Mission records:  {mission.shape[0]:,} rows x {mission.shape[1]} columns')
print(f'AED locations:    {aed.shape[0]:,} rows x {aed.shape[1]} columns')

## 2.2 Quantitative Quality Checks

In [ ]:
checks = {
    'Dispatch validity rate': f"{dispatch['dispatch_valid'].mean():.1%}",
    'Response time in range (0.5-240 min)': f"{mission['response_min'].between(0.5, 240).mean():.1%}",
    'Missions within Belgium bbox': f"{(mission['latitude'].between(49, 52) & mission['longitude'].between(2, 7)).mean():.1%}",
    'AEDs within Belgium bbox': f"{(aed['latitude'].between(49, 52) & aed['longitude'].between(2, 7)).mean():.1%}",
}

qa_df = pd.DataFrame(list(checks.items()), columns=['Quality Metric', 'Result'])
print(qa_df)

## 2.3 Missingness Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mission record missingness
miss_mission = mission.isnull().mean().sort_values(ascending=False).head(15)
miss_mission.plot(kind='barh', ax=axes[0], color='#e74c3c', edgecolor='white')
axes[0].set_title('Mission Records: Top 15 Missing Columns', fontweight='bold')
axes[0].set_xlabel('Fraction Missing')
axes[0].invert_yaxis()

# Response time distribution
mission['response_min'].clip(upper=120).hist(bins=60, ax=axes[1], color='#3498db', edgecolor='white')
axes[1].axvline(mission['response_min'].median(), color='red', linestyle='--', label=f"Median: {mission['response_min'].median():.1f} min")
axes[1].set_title('Response Time Distribution (clipped at 120 min)', fontweight='bold')
axes[1].set_xlabel('Response Time [min]')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

## Summary

**Data quality design principles:**

- Records are not force-filled; only explicitly valid records proceed to modeling
- Mission-level response uses the earliest valid vector arrival as the performance proxy
- All final modeling samples carry traceable quality flags

The cleaned datasets feed directly into **Notebook 03** (Geospatial Baseline) and **Notebook 04** (Predictive Modeling).